# 🩺 Data Preparation for Epidemiologists Using Python
**Epidemiology and Python — YouTube Tutorial Series**  
Based on: *Predicting Changes in Systolic and Diastolic Blood Pressure of Hypertensive Patients in Indonesia*  
Dataset: 52,350 hypertensive patients | Author: Desy Nuryunarsih

---

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns

## 📂 Step 2: Open the Dataset

In [ ]:
df = pd.read_csv('hypertension_52350.csv')

In [ ]:
df.head()

## 🔍 Step 3: Know the Data

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns.tolist()

## ❓ Step 4: Visualise Missing Values (missingno)

In [ ]:
msno.matrix(df)

In [ ]:
msno.heatmap(df)

## 📊 Step 5: Missing Value Percentage

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2).sort_values(ascending=False)

## 🗑️ Step 6: Drop Rows with Missing Values (dropna)

In [ ]:
df = df.dropna()
df.shape

## ✂️ Step 7: Select Variables We Need

In [ ]:
cols_to_keep = [
    'Diagnosis', 'Age', 'AGE of HTN', 'Sex', 'BMI(kg/m^2)',
    'SBP', 'DBP', 'Smoking/Naswar', 'Physical Activity',
    'Treatment', 'Family HTN', 'Family CVD',
    'Diet plan', 'meals/day', 'Income', 'Job',
    'Headache', 'Dizziness', 'Chest Pain', 'Palpitations',
    'Province', 'Date of Visit'
]

df = df[cols_to_keep]
df.shape

## 🔢 Step 8: Convert Columns to Numeric

In [ ]:
df['SBP']         = pd.to_numeric(df['SBP'],         errors='coerce')
df['DBP']         = pd.to_numeric(df['DBP'],         errors='coerce')
df['BMI(kg/m^2)'] = pd.to_numeric(df['BMI(kg/m^2)'], errors='coerce')
df['Age']         = pd.to_numeric(df['Age'],         errors='coerce')
df['AGE of HTN']  = pd.to_numeric(df['AGE of HTN'],  errors='coerce')

## 🏷️ Step 9: Make Categorical Variables

In [ ]:
cat_cols = [
    'Sex', 'Smoking/Naswar', 'Physical Activity', 'Treatment',
    'Family HTN', 'Family CVD', 'Diet plan', 'Job',
    'Headache', 'Dizziness', 'Chest Pain', 'Palpitations', 'Diagnosis'
]

df[cat_cols] = df[cat_cols].astype('category')
df.dtypes

## 📅 Step 10: Convert Date Column to Datetime

In [ ]:
df['Date of Visit'] = pd.to_datetime(df['Date of Visit'], errors='coerce')
df['Date of Visit'].dtype

## 🩹 Step 11: Fill Missing Values (fillna) for Selected Variables

In [ ]:
df['SBP']         = df['SBP'].fillna(df['SBP'].median())
df['DBP']         = df['DBP'].fillna(df['DBP'].median())
df['BMI(kg/m^2)'] = df['BMI(kg/m^2)'].fillna(df['BMI(kg/m^2)'].median())

## 📈 Step 12: Descriptive Statistics

In [ ]:
df.describe()

In [ ]:
df.describe(include='category')

## 👥 Step 13: GroupBy

In [ ]:
df.groupby('Sex')[['SBP', 'DBP', 'BMI(kg/m^2)']].mean().round(2)

In [ ]:
df.groupby('Diagnosis')[['SBP', 'DBP', 'Age']].mean().round(2)

## 🔀 Step 14: Cross Tabulation

In [ ]:
pd.crosstab(df['Sex'], df['Diagnosis'])

In [ ]:
pd.crosstab(df['Smoking/Naswar'], df['Diagnosis'], normalize='index').round(2)

## 🔁 Step 15: Check & Remove Duplicate Rows

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated(keep=False)].head(10)

In [ ]:
df = df.drop_duplicates()
df.shape

## 📦 Step 16: Outlier Detection — Boxplots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
df.boxplot(column='SBP',         ax=axes[0])
df.boxplot(column='DBP',         ax=axes[1])
df.boxplot(column='BMI(kg/m^2)', ax=axes[2])
plt.tight_layout()
plt.show()

## 🗺️ Step 17: Mapping — Merge with External CSV

In [ ]:
province_map = pd.read_csv('province_mapping.csv')
province_map

In [ ]:
df = df.merge(province_map, on='Province', how='left')
df[['Province', 'Region', 'Island', 'Population_million', 'Health_Facility_per_100k']].head()

## 📊 Step 18: Descriptive Graphs

### Distribution of SBP and DBP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['SBP'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Systolic Blood Pressure')
axes[0].set_xlabel('SBP (mmHg)')
df['DBP'].hist(bins=30, ax=axes[1], color='tomato', edgecolor='white')
axes[1].set_title('Distribution of Diastolic Blood Pressure')
axes[1].set_xlabel('DBP (mmHg)')
plt.tight_layout()
plt.show()

### Diagnosis Count by Sex

In [ ]:
pd.crosstab(df['Diagnosis'], df['Sex']).plot(
    kind='bar', figsize=(8, 4), colormap='Set2', edgecolor='white'
)
plt.title('Hypertension Diagnosis by Sex')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

### Smoking Status Distribution

In [ ]:
df['Smoking/Naswar'].value_counts().plot(
    kind='bar', figsize=(7, 4),
    color=['#2ecc71', '#e67e22', '#e74c3c'], edgecolor='white'
)
plt.title('Smoking Status of Patients')
plt.xlabel('Smoking Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Physical Activity Distribution

In [ ]:
df['Physical Activity'].value_counts().plot(
    kind='pie', autopct='%1.1f%%', figsize=(6, 6),
    colors=['#3498db', '#f39c12', '#e74c3c']
)
plt.title('Physical Activity Level')
plt.ylabel('')
plt.show()

### Mean SBP by Province

In [ ]:
df.groupby('Province')['SBP'].mean().sort_values().plot(
    kind='barh', figsize=(8, 5), color='steelblue', edgecolor='white'
)
plt.title('Mean Systolic Blood Pressure by Province')
plt.xlabel('Mean SBP (mmHg)')
plt.tight_layout()
plt.show()

## 💾 Step 19: Save Clean Dataset as Parquet

In [ ]:
df.to_parquet('hypertension_clean.parquet', index=False)
print('✅ Saved as Parquet — ready for analysis!')

---
## ✅ Data Preparation Complete!
**Clean dataset:** `hypertension_clean.parquet`  
**Next:** Machine Learning — Predicting Changes in SBP and DBP  

*Epidemiology and Python | Desy Nuryunarsih*